<a id="top"></a>

# Demo: three outcomes, including a hallucinated call

```yaml
title: "Demo: three outcomes, including a hallucinated call"
keywords: three outcomes, hallucinated tool call, validation, application validates, tool_choice, haiku contrast, non-determinism, langchain, tool_calls, teacher demo
estimated duration: 13
```

> **Lesson:** L07. Teacher demo — Demo 4 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L07/demos_or_activities.md). Makes **live** Claude calls — set `ANTHROPIC_API_KEY` before running. Model behavior **varies**: if Sonnet is too well-behaved on P3, re-run on **Haiku 4.5**, or use the offline fallback (last cell) to force the validation path. Clear outputs before committing.
>
> **Model-agnostic by design.** Both models are LangChain `ChatAnthropic` handles bound with the same tool; every reply exposes `.tool_calls` regardless of provider. The key loads through the config seam (`require_anthropic_key`); we never hard-code it.
>
> The point to land: handing the model a tool has **three observable outcomes** — calls it, skips it, or calls it with bad arguments — and the **application**, not the model, catches the bad case. A clean `.tool_calls` *shape* is not a valid *value*; the schema is a contract about shape, not a guarantee about behavior.

## Contents

- [1. Setup — the tool, plus a visible validator](#1-setup--the-tool-plus-a-visible-validator)
- [2. Outcome one — a clean tool call (P1)](#2-outcome-one--a-clean-tool-call-p1)
- [3. Outcome two — the model answers without the tool (P2)](#3-outcome-two--the-model-answers-without-the-tool-p2)
- [4. Outcome three — a malformed / hallucinated call (P3)](#4-outcome-three--a-malformed--hallucinated-call-p3)
- [5. Fallback — force the validation path by hand](#5-fallback--force-the-validation-path-by-hand)
- [6. Takeaways](#6-takeaways)

## 1. Setup — the tool, plus a visible validator

The same `calculator`, plus a `dispatch` helper whose validation step is **visible**: a bad call is *rejected*, not crashed-on. Two bound models ready — Sonnet 4.6 (anchor) and Haiku 4.5 (the smaller-model contrast). Live cells are gated on `LIVE`; the offline fallback and `dispatch` run with no key.

In [ ]:
from typing import Any

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

from fluffy_potato_curriculum.common.config import get_settings, require_anthropic_key

SONNET = "claude-sonnet-4-6"  # course anchor
HAIKU = "claude-haiku-4-5"  # smaller model — fumbles tool selection / args more often
LIVE = bool(get_settings().anthropic_api_key)


# The ONE tool that carries all four demos: a deterministic calculator.
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression and return the exact result as a string.

    Use this whenever the user asks for the result of a calculation. Restricted to
    digits and the operators + - * / ( ) . and whitespace so a hallucinated
    expression cannot run arbitrary code. Raises ValueError on anything else — that
    rejection is the whole point of this demo.
    """
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    # eval is safe here ONLY because the character whitelist above blocks names,
    # attributes, and calls. Never eval untrusted input without such a guard.
    return str(eval(expression))


TOOLS = [calculator]


def dispatch(call: dict[str, Any]) -> str:
    """Validate a tool call (a {"name", "args", "id"} dict) and run it, or reject it loudly.

    The application owns three guards — unknown tool name, missing argument, and a
    bad value the tool itself refuses. The model's clean-looking `.tool_calls` dict
    passed none of them automatically.
    """
    if call["name"] != "calculator":
        return f"REJECTED: no such tool {call['name']!r} (the model invented it)"
    args = call["args"]
    if "expression" not in args:
        return f"REJECTED: missing required argument 'expression' (got {args!r})"
    try:
        return f"OK: calculator({args['expression']!r}) = {calculator(args['expression'])}"
    except ValueError as exc:
        return f"REJECTED: {exc}"


# Bind the tool to each model once; every reply then exposes `.tool_calls`.
if LIVE:
    key = require_anthropic_key()
    BOUND = {
        SONNET: ChatAnthropic(model=SONNET, api_key=key, max_tokens=400).bind_tools(TOOLS),
        HAIKU: ChatAnthropic(model=HAIKU, api_key=key, max_tokens=400).bind_tools(TOOLS),
    }

print("setup ready (LIVE =", LIVE, ")")

[↑ Back to top](#top)

## 2. Outcome one — a clean tool call (P1)

Arithmetic the model should defer to the tool. A non-empty `.tool_calls`, valid args, a real result.

In [ ]:
if LIVE:
    resp = BOUND[SONNET].invoke([HumanMessage("What is 47,219 multiplied by 8,803?")])
    print("tool calls:", [c["name"] for c in resp.tool_calls])
    for call in resp.tool_calls:
        print("dispatch ->", dispatch(call))
else:
    print("offline: set ANTHROPIC_API_KEY to run this live cell")

[↑ Back to top](#top)

## 3. Outcome two — the model answers without the tool (P2)

A trivial sum the model already owns. Usually it answers directly — `.tool_calls` is empty, only `.text`. A registered tool is an **option**, never an obligation (adding a tool to a task the model already owns is wasted overhead — a forward-link to [L08](L0702_lecture.md)).

In [ ]:
if LIVE:
    resp = BOUND[SONNET].invoke([HumanMessage("What is 2 + 2?")])
    print("tool calls:", [c["name"] for c in resp.tool_calls])
    print("text:", repr(resp.text))
    print("did the model call the tool?", bool(resp.tool_calls))
else:
    print("offline: set ANTHROPIC_API_KEY to run this live cell")

[↑ Back to top](#top)

## 4. Outcome three — a malformed / hallucinated call (P3)

A prompt that nudges the model toward a bad argument (a non-numeric token in the expression). The model may emit a tool call whose arguments are malformed — and the **application's validation catches it**. If Sonnet sanitizes cleanly, Haiku often doesn't.

In [ ]:
P3 = "Use the calculator to work out the average of these three: 12, 19, and twenty."

if LIVE:
    for model_name in (SONNET, HAIKU):
        resp = BOUND[model_name].invoke([HumanMessage(P3)])
        print(f"=== {model_name} ===")
        print("  tool calls:", [c["name"] for c in resp.tool_calls])
        for call in resp.tool_calls:
            print("  proposed args:", call["args"])
            print("  dispatch ->", dispatch(call))
else:
    print("offline: set ANTHROPIC_API_KEY to run this live cell (or use the fallback below)")

[↑ Back to top](#top)

## 5. Fallback — force the validation path by hand

If neither model fumbles on the day, hand-build a few bad tool calls (the same `{"name", "args", "id"}` dicts LangChain would produce) and feed them to `dispatch`. The teaching point is the application's **response** to a bad call, which you can show regardless of whether the model cooperated. This cell needs no key.

In [ ]:
# Hand-built tool calls that mimic three different hallucinations. These are the
# same plain dicts a real reply's `.tool_calls` would contain.
bad_calls: list[dict[str, Any]] = [
    {"name": "calculator", "args": {"expression": "twenty + 1"}, "id": "c1"},  # non-numeric
    {"name": "calculator", "args": {}, "id": "c2"},  # missing arg
    {"name": "wikipedia", "args": {"query": "geese"}, "id": "c3"},  # invented tool
]
for call in bad_calls:
    print(dispatch(call))

[↑ Back to top](#top)

## 6. Takeaways

- The model can **hallucinate** a tool call: wrong types, missing required args, invented args, even a tool name that doesn't exist. The schema stopped **none** of it at generation time — **the application validates; the model proposes.**
- LangChain hands you a tidy `.tool_calls` dict, but a clean *shape* is not a valid *value* — the arguments inside can still be nonsense. Validation stays your job.
- Showing one hallucination is worth ten clean runs — the L07 analogue of L06's tag-violation moment.
- Tool calls are **not deterministic**: same prompt + schema can call, skip, or pass different arguments across runs. That is *why* validation is not optional.
- `tool_choice` (via `bind_tools(..., tool_choice=...)`) can *bias* the decision, but forcing a call still does not guarantee well-formed arguments. Designing the error *response* is [L08](L0702_lecture.md)'s job, not L07's.

[↑ Back to top](#top)